In [1]:
%load_ext autoreload
%autoreload 2

# 长上下文解码过程中的注意力数据分析

## 参考模型

- Llama3 系列
    - Llama-3.1-8b (130k)、Llama-3-8b-1M (1048k)
    - GQA
    - 32 Layers、8 KVHeads、32 Attention Heads (32/8=4 query groups)

## 收集指标

- **Key、Value** Tensor per **Layer** per **Group** per **Head**
    - w、w/o RoPE
- Attention Score per **Head** per **Layer**
- **Query** Tensor during **decoding**


In [4]:
from transformers import AutoConfig

llama3_path = "/home/yexuming/.cache/modelscope/hub/models/LLM-Research/Meta-Llama-3___1-8B"
llama3_cfg = AutoConfig.from_pretrained(llama3_path)
llama3_cfg

/home/yexuming/bwl/research/llm_utils/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "bfloat16",
  "eos_token_id": 128001,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_scaling": {
    "factor": 8.0,
    "high_freq_factor": 4.0,
    "low_freq_factor": 1.0,
    "original_max_position_embeddings": 8192,
    "rope_type": "llama3"
  },
  "rope_theta": 500000.0,
  "tie_word_embeddings": false,
  "transformers_version": "4.57.1",
  "use_cache": true,
  "vocab_size": 128256
}

# Ruler Report

Ruler Synthetic Datasets 64k inputs

> run aime_sweep.py first to generate tensor files

In [1]:
import json
import random

def sample_from_jsonl(jsonl_path, sample_num=1):
    """从指定路径解析 .jsonl 数据集文件，并随机采样一条样本"""
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    sample_lines = random.sample(lines, sample_num)
    samples = [json.loads(line) for line in sample_lines]
    return samples

jsonl_path = "/home/yexuming/datasets/ruler/llama-3/65536/fwe/validation.jsonl"
sample = sample_from_jsonl(jsonl_path)
sample

[{'index': 4,
  'input': "<|begin_of_text|><|start_header_id|>system<|end_header_id|>You are a helpful assistant<|eot_id|><|start_header_id|>user<|end_header_id|>Read the following coded text and track the frequency of each coded word. Find the three most frequently appeared coded words. ... ctvoev ... uakvji uakvji uakvji qbqtqy ... ... qbqtqy ... ... ... ... ... uakvji ... ... ... qvseye ctvoev ... ctvoev ... ... ... ... ... ... cblveh ... bctqxm ... uakvji ... xuzyec qbqtqy ... ... cblveh ... uakvji uakvji ... mdlmci ... ... ... ... uakvji ... uakvji ... akmdqd ... ... uakvji cblveh qbqtqy cblveh ... uakvji ... pajzgj ... ... qbqtqy ctvoev ... uakvji ydmmwz uakvji ... ... qbqtqy ... uakvji ... ... ... pizpyu ... ... cblveh uakvji ... ctvoev uakvji ... ... ... uakvji ... ... qbqtqy qbqtqy qbqtqy ... ... ... ... uakvji ... xuzyec ctvoev ... ... ... ... ... ... qbqtqy ... ... ... ... engpfq ... qbqtqy ... ... xuzyec ... ... ... uakvji ... ... ... ... ... uakvji ... ... dacbot ... bctqx

In [ ]:
from transformers import AutoTokenizer

prompt = sample[0]["input"]
toker = AutoTokenizer.from_pretrained(llama3_path)
inputs = toker(prompt, return_tensors="pt").to("cuda:0")
print(f"Actual input length: f{inputs.input_ids.shape}")

In [ ]:
from transformers import AutoModelForCausalLM
llama3_model = AutoModelForCausalLM.from_pretrained(llama3_path)
llama3_model

Loading checkpoint shards: 100%|██████████| 4/4 [00:01<00:00,  2.64it/s]


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4096,), eps=1e-05)
  